## Import necessary packages, remember to pip install any missing packages.

In [ ]:
# Pip install the required libraries (uncomment if running for the first time). I think you already have these installed, but just in case:
#pip install -q transformers==5.4.0
#pip install -q torch==2.11.0
#pip install -q seaborn==0.13.2


import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
from transformers import BertTokenizer
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
import datetime

from tensorflow.keras.preprocessing.sequence import pad_sequences

from torch.utils.data import Dataset, TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup



## Load the data

In [ ]:
df_fake = pd.read_csv("C:/Users/HJ25ZC/OneDrive - Aalborg Universitet/Documents/Kurser/AI og data/Fake_True/Fake.csv")
df_real = pd.read_csv("C:/Users/HJ25ZC/OneDrive - Aalborg Universitet/Documents/Kurser/AI og data/Fake_True/True.csv")
df_fake["category"] = 1
df_real["category"] = 0

# First half: real news, second half: fake news
df = pd.concat([df_real[:100], df_fake[:100]], ignore_index=True) # Only 200 samples are loaded right now to speed up the code.
df = df.reset_index(drop=True)


## Exercise 1
Create embeddings of the dataset, using the pretrained tokenizer. 
Visualise these embeddings using TSNE plots.

In [ ]:
from transformers import AutoTokenizer
from sklearn.manifold import TSNE

### ==================== CONFIGURATION PARAMETERS ====================
embed_dim = 64          # Embedding dimension (experiment with different values)
sample_size = 200        # Number of documents to use (smaller = faster computation)
### ====================================================================

### Set up tokenizer and embedding ###
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

### Create embeddings layer ###
embedding = nn.Embedding(tokenizer.vocab_size, embed_dim)
print(f"✓ Embedding layer created with dimension: {embed_dim}")
print(embedding)

### Extract text from dataframe (combine title and text columns) ###
df_sample = df.head(sample_size)
print(f"✓ Using first {sample_size} documents for analysis")

### Create embeddings for each document ###
embeddings_list = []
for idx, row in df_sample.iterrows():
    combined_text = row['title'] + ' ' + row['text']
    tokens = tokenizer(combined_text, return_tensors='pt', truncation=True, max_length=128)
    embedded = embedding(tokens['input_ids'])
    # Average pooling to get document-level embedding
    doc_embedding = embedded.mean(dim=1)
    embeddings_list.append(doc_embedding.squeeze().detach())

### Stack all embeddings into a single tensor ###
embeddings_tensor = torch.stack(embeddings_list)  # Shape: (num_docs, embed_dim)
print(f"Embeddings tensor shape: {embeddings_tensor.shape}")

### Visualize embeddings using t-SNE ###
print("\nGenerating t-SNE visualization...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings_tensor.numpy())

### Create scatter plot ###
plt.figure(figsize=(12, 8))
categories = df_sample['category'].values
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=categories, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Visualization of Document Embeddings\n(Blue=Real, Yellow=Fake)')
cbar = plt.colorbar(scatter)
cbar.set_label('Category (0=Real, 1=Fake)')
plt.tight_layout()
plt.show()

### Display sample embeddings ###
print(f"\n--- Sample Documents and Embeddings ---")
for i in range(min(5, sample_size)):
    print(f"Doc {i}: Category={df_sample.iloc[i]['category']} (0=Real, 1=Fake)")
    print(f"  Title: {df_sample.iloc[i]['title'][:60]}...")
    print(f"  Embedding norm: {embeddings_tensor[i].norm():.4f}")
    print(f"  t-SNE coords: ({embeddings_2d[i, 0]:.3f}, {embeddings_2d[i, 1]:.3f})\n")

## Exercise 2
Calculate attention between different sentences in the dataset (fake, fake), (real, real), and (real, fake)
Visualise the attention using plt.imshow

In [ ]:
### Calculate Single-Head Attention (Baseline) ###
similarity = embeddings_tensor @ embeddings_tensor.transpose(0, 1)
print(f"Similarity matrix shape: {similarity.shape}")
print(f"Similarity matrix (first 5x5):\n{similarity[:5, :5]}")

### Normalize the Similarity Scores ###
print(f"\nPre-normalization Variance: {similarity.var()}")
similarity_norm = similarity / (embed_dim**0.5)
print(f"Normalized Similarity Variance: {similarity_norm.var()}")

### Apply Softmax to get Attention Weights ###
attention_mat = similarity_norm.softmax(dim=-1)

### Verify each row sums to 1 ###
summed_attention_mat = attention_mat.sum(dim=-1)
print(f"\nAll rows sum to 1: {torch.allclose(summed_attention_mat, torch.ones_like(summed_attention_mat))}")

### Compute weighted context vectors using attention ###
context_vectors = attention_mat @ embeddings_tensor
print(f"Context vectors shape: {context_vectors.shape}")

### Visualize attention weights as heatmap (full matrix) ###
plt.figure(figsize=(14, 13), dpi=150)
plt.imshow(attention_mat.detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Document Index (Real News: 0-99, Fake News: 100-199)')
plt.ylabel('Document Index (Real News: 0-99, Fake News: 100-199)')
plt.title('Single-Head Attention Matrix: How Each Document Attends to All Others\n(Expect higher attention within same category blocks)')

# Add dividing lines to show real vs fake news boundaries
num_real = 100
plt.axhline(y=num_real - 0.5, color='red', linewidth=2, linestyle='--', label='Real/Fake boundary')
plt.axvline(x=num_real - 0.5, color='red', linewidth=2, linestyle='--')

# Add text annotations for clarity
plt.text(50, -15, 'REAL NEWS', ha='center', fontsize=12, fontweight='bold', color='blue')
plt.text(150, -15, 'FAKE NEWS', ha='center', fontsize=12, fontweight='bold', color='red')
plt.text(-25, 50, 'REAL NEWS', ha='right', fontsize=12, fontweight='bold', color='blue', rotation=90)
plt.text(-25, 150, 'FAKE NEWS', ha='right', fontsize=12, fontweight='bold', color='red', rotation=90)

plt.legend(loc='upper right')
plt.subplots_adjust(top=0.50)
plt.tight_layout()
plt.show()

# Print statistics about attention within and across categories
real_real_attention = attention_mat[:num_real, :num_real].mean().item()
fake_fake_attention = attention_mat[num_real:, num_real:].mean().item()
real_fake_attention = attention_mat[:num_real, num_real:].mean().item()
fake_real_attention = attention_mat[num_real:, :num_real].mean().item()

print("\n=== Attention Statistics ===")
print(f"Real news attending to Real news: {real_real_attention:.4f}")
print(f"Fake news attending to Fake news: {fake_fake_attention:.4f}")
print(f"Real news attending to Fake news: {real_fake_attention:.4f}")
print(f"Fake news attending to Real news: {fake_real_attention:.4f}")

## Multi-headed attention
Change the number of heads and the dimension of the embeddings and see what happens.


In [ ]:
### Multi-Head Attention ###
print("=== Multi-Head Attention ===\n")

# CONFIGURABLE PARAMETER - Change this to experiment with different numbers of heads
num_heads = 4

# Validate that embed_dim is defined and num_heads divides it evenly
print(f"Current embed_dim: {embed_dim}")
print(f"Requested num_heads: {num_heads}\n")

if embed_dim % num_heads != 0:
    print(f" ERROR: embed_dim ({embed_dim}) is not divisible by num_heads ({num_heads})")
    print(f"   embed_dim = {embed_dim}, num_heads = {num_heads}")
    print(f"   {embed_dim} / {num_heads} = {embed_dim / num_heads} (not an integer)\n")
    
    # Find valid divisors
    valid_heads = [i for i in range(1, embed_dim + 1) if embed_dim % i == 0]
    print(f"✓ Valid options for num_heads: {valid_heads}")
    print(f"\n  To fix: change num_heads to one of the values above (e.g., {valid_heads[-2]})")
    raise ValueError(f"num_heads ({num_heads}) must divide embed_dim ({embed_dim}) evenly")

head_dim = embed_dim // num_heads
print(f"✓ Number of heads: {num_heads}")
print(f"✓ Dimension per head: {head_dim}")
print(f"✓ Total: {num_heads} × {head_dim} = {num_heads * head_dim}\n")

### Project embeddings into multiple heads ###
# Reshape embeddings tensor to split into heads
# Shape: (num_docs, embed_dim) -> (num_docs, num_heads, head_dim)
embeddings_heads = embeddings_tensor.view(embeddings_tensor.shape[0], num_heads, head_dim)

### Calculate attention for each head separately ###
attention_heads = []
context_vectors_heads = []

for head_idx in range(num_heads):
    # Get embeddings for this head
    head_embeddings = embeddings_heads[:, head_idx, :]  # Shape: (num_docs, head_dim)
    
    # Calculate similarity matrix for this head
    head_similarity = head_embeddings @ head_embeddings.transpose(0, 1)
    
    # Normalize similarity scores
    head_similarity_norm = head_similarity / (head_dim**0.5)
    
    # Apply softmax to get attention weights for this head
    head_attention = head_similarity_norm.softmax(dim=-1)
    
    # Compute context vectors for this head
    head_context = head_attention @ head_embeddings
    
    attention_heads.append(head_attention)
    context_vectors_heads.append(head_context)

### Concatenate context vectors from all heads ###
multi_head_context = torch.cat(context_vectors_heads, dim=1)
print(f"Multi-head context vectors shape: {multi_head_context.shape}")

### Visualize attention from individual heads ###
# Dynamically create subplots based on number of heads
num_cols = min(4, num_heads)  # Max 4 columns
num_rows = (num_heads + num_cols - 1) // num_cols  # Ceiling division
fig, axes = plt.subplots(num_rows, num_cols, figsize=(5*num_cols, 5*num_rows), dpi=150)
axes = axes.flatten()

for head_idx in range(num_heads):
    ax = axes[head_idx]
    attention_matrix = attention_heads[head_idx].detach().numpy()
    im = ax.imshow(attention_matrix, cmap='viridis', aspect='auto')
    ax.set_xlabel('Document Index')
    ax.set_ylabel('Document Index')
    ax.set_title(f'Head {head_idx + 1} Attention Matrix')
    plt.colorbar(im, ax=ax, label='Attention Weight')

# Hide unused subplots
for idx in range(num_heads, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### Visualize average attention across all heads ###
average_attention = torch.stack(attention_heads, dim=0).mean(dim=0)
print(f"Average attention shape: {average_attention.shape}")

plt.figure(figsize=(12, 10), dpi=150)
plt.imshow(average_attention.detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Average Attention Weight')
plt.xlabel('Document Index')
plt.ylabel('Document Index')
plt.title('Multi-Head Attention: Average Across All Heads')
plt.tight_layout()
plt.show()

### Verify attention properties for multi-head ###
print("\nMulti-Head Attention Properties:")
try:
    for head_idx in range(num_heads):
        summed = attention_heads[head_idx].sum(dim=-1)
        print(f"Head {head_idx + 1} - All rows sum to 1: {torch.allclose(summed, torch.ones_like(summed))}")
except Exception as e:
    print(f"❌ Error verifying attention properties: {e}")
    print(f"   Number of attention_heads computed: {len(attention_heads)}")
    print(f"   num_heads requested: {num_heads}")


## Exercise 3
Play around with the code and see what happens if you change the dimensionality of the embeddings.
Explain what is visualised.
Show what happens with the embeddings after implementing positional encoding, by using TSNE plots.
(Optional): Look into other positional embedding methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
 
plt.rcParams.update({'font.size': 18})
 
def sinusoidal_positional_embedding(dim, positions):
    """
    Generate sinusoidal positional embeddings.
 
    Args:
    dim: int, dimension of the embeddings.
    positions: numpy array of shape (num_positions,), positions for which to generate the embeddings.
 
    Returns:
    numpy array of shape (num_positions, dim), representing the sinusoidal positional embeddings.
    """
    i = np.arange(dim // 2).reshape(1, -1)
    pos = positions.reshape(-1, 1)
    emb = np.zeros((len(positions), dim))
    emb[:, 0::2] = np.sin(pos / 10000 ** ((2 * i) / dim))
    emb[:, 1::2] = np.cos(pos / 10000 ** ((2 * i + 1) / dim))
    return emb
 
# Define parameters
dim = [128, 128, 256]  # Dimension of the embeddings
num_positions = [32, 64, 32]  # Number of positions
 
 
# Create subplots
fig, axs = plt.subplots(3, 1, figsize=(16, 12), dpi=200)
 
# Plot each embedding
for i, ax in enumerate(axs):
    positions = np.arange(num_positions[i])  # Positions to plot embeddings for
 
    # Generate sinusoidal positional embeddings for each position
    embeddings = sinusoidal_positional_embedding(dim[i], positions)
    ax.imshow(embeddings, cmap='RdBu', aspect='auto', interpolation='nearest')
    ax.set_xlabel('Encoding Dimension')
    ax.set_ylabel('Encoding Position')
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
 
 
# Add colorbar to the last subplot
cbar = axs[-1].figure.colorbar(axs[-1].images[0], ax=axs, label='Encoding Value')
cbar.set_ticks([-1.00, -0.75, -0.50, -0.25, 0.00, 0.25, 0.50, 0.75, 1.00])  # Set specific ticks
 
 
# # Show the plot
plt.show()
 
def calculate_l2_distance(embeddings):
    """
    Calculate the L2 distance between all embeddings.
 
    Args:
    embeddings: numpy array of shape (num_positions, dim), representing the embeddings.
 
    Returns:
    numpy array of shape (num_positions, num_positions), representing the L2 distance matrix.
    """
    num_positions = embeddings.shape[0]
    distances = np.zeros((num_positions, num_positions))
    for i in range(num_positions):
        for j in range(num_positions):
            distances[i, j] = np.linalg.norm(embeddings[i] - embeddings[j], ord = 2)
    return distances
 
 
dim = 32  # Dimension of the embeddings
num_positions = 32  # Number of positions
positions = np.arange(num_positions)  # Positions to plot embeddings for
 
embeddings = sinusoidal_positional_embedding(dim, positions)
 
# Calculate L2 distance
distances = calculate_l2_distance(embeddings)
plt.figure(figsize=(10, 8), dpi = 200)
# Plot L2 distance heatmap
plt.imshow(distances, cmap='RdBu', aspect='auto', interpolation='nearest')
plt.ylabel("Encoding Position")
plt.xlabel("Encoding Position")
 
# Add colorbar to the heatmap
plt.colorbar(label="Distance")
plt.tick_params(axis='x', which='both', bottom=False, top=True, labelbottom=False, labeltop=True)
 
 
 
def calculate_dot_product(embeddings):
    num_positions = embeddings.shape[0]
    dot = np.zeros((num_positions, num_positions))
    for i in range(num_positions):
        for j in range(num_positions):
            dot[i, j] = np.dot(embeddings[i], embeddings[j])
    return dot
 
 
dim = 256  # Dimension of the embeddings
num_positions = 32  # Number of positions
positions = np.arange(num_positions)  # Positions to plot embeddings for
 
embeddings = sinusoidal_positional_embedding(dim, positions)
 
# Calculate L2 distance
dot = calculate_dot_product(embeddings)
plt.figure(figsize=(10, 8), dpi = 200)
# Plot L2 distance heatmap
plt.imshow(dot, cmap='RdBu', aspect='auto', interpolation='nearest')
plt.ylabel("Encoding Position")
plt.xlabel("Encoding Position")
 
# Add colorbar to the heatmap
plt.colorbar(label="Dot-Product")
plt.tick_params(axis='x', which='both', bottom=False, top=True, labelbottom=False, labeltop=True)

## Exercise 3b
Experiment with perplexity for TSNE plots. What happens when it is decreased or increased?

In [ ]:
### Positional Encoding Based on Document Data ###
print("=== Positional Encoding on Document Data ===\n")

### Generate positional embeddings for our documents ###
num_docs = embeddings_tensor.shape[0]
pos_dim = embed_dim  # Use same dimension as embeddings

### Create positional encodings for document positions ###
positional_encodings = sinusoidal_positional_embedding(pos_dim, np.arange(num_docs))
positional_encodings_tensor = torch.from_numpy(positional_encodings).float()

print(f"Positional encodings shape: {positional_encodings_tensor.shape}")
print(f"Document embeddings shape: {embeddings_tensor.shape}\n")

### Calculate L2 distance between positional encodings ###
pos_distances = calculate_l2_distance(positional_encodings)

### Calculate dot product between positional encodings ###
pos_dot = calculate_dot_product(positional_encodings)

### Visualize positional encoding properties ###
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=150)

# L2 Distance heatmap
im1 = axes[0].imshow(pos_distances, cmap='RdBu', aspect='auto', interpolation='nearest')
axes[0].set_ylabel("Document Position")
axes[0].set_xlabel("Document Position")
axes[0].set_title(f'L2 Distance Between Positional Encodings\n(Dimension={pos_dim}, Num Documents={num_docs})')
plt.colorbar(im1, ax=axes[0], label="Distance")

# Dot product heatmap
im2 = axes[1].imshow(pos_dot, cmap='RdBu', aspect='auto', interpolation='nearest')
axes[1].set_ylabel("Document Position")
axes[1].set_xlabel("Document Position")
axes[1].set_title(f'Dot Product Between Positional Encodings\n(Dimension={pos_dim}, Num Documents={num_docs})')
plt.colorbar(im2, ax=axes[1], label="Dot Product")

plt.tight_layout()
plt.show()

### Combine document embeddings with positional encodings ###
combined_embeddings = embeddings_tensor + positional_encodings_tensor

### Compare original embeddings vs. positional encoding effect ###
print("Statistics Comparison:")
print(f"Original embeddings - Mean norm: {embeddings_tensor.norm(dim=1).mean():.4f}, Std: {embeddings_tensor.norm(dim=1).std():.4f}")
print(f"Positional encodings - Mean norm: {positional_encodings_tensor.norm(dim=1).mean():.4f}, Std: {positional_encodings_tensor.norm(dim=1).std():.4f}")
print(f"Combined embeddings - Mean norm: {combined_embeddings.norm(dim=1).mean():.4f}, Std: {combined_embeddings.norm(dim=1).std():.4f}\n")

### Visualize original vs combined embeddings using t-SNE ###
tsne = TSNE(n_components=2, random_state=42, perplexity=2, max_iter=1000)
combined_2d = tsne.fit_transform(combined_embeddings.numpy())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original embeddings
categories = df_sample['category'].values
scatter1 = axes[0].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                          c=categories, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_xlabel('t-SNE Component 1')
axes[0].set_ylabel('t-SNE Component 2')
axes[0].set_title('Document Embeddings (No Positional Encoding)')
plt.colorbar(scatter1, ax=axes[0], label='Category (0=Real, 1=Fake)')

# Combined embeddings with positional encoding
scatter2 = axes[1].scatter(combined_2d[:, 0], combined_2d[:, 1], 
                          c=categories, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_xlabel('t-SNE Component 1')
axes[1].set_ylabel('t-SNE Component 2')
axes[1].set_title('Document Embeddings + Positional Encoding')
plt.colorbar(scatter2, ax=axes[1], label='Category (0=Real, 1=Fake)')

plt.tight_layout()
plt.show()

print(f"Positional encoding successfully added to {num_docs} documents!")

## (Optional) Exercise 4 - Classification with BERT
This is a very simple example of a classification task using the BERT model. Load the checkpoint available on moodle for this.

In [ ]:
# Load pretrained BERT checkpoint and validate on data with confusion matrix & attention visualization

import torch
import torch.nn.functional as F
from transformers import BertForSequenceClassification, BertTokenizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load checkpoint
checkpoint_path = 'Bert_checkpoints/checkpoint_epoch_0_batch_499.pt'
# Use eager attention to support output_attentions
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', 
    num_labels=2,
    attn_implementation='eager'
)
checkpoint = torch.load(checkpoint_path, map_location=device)

# The checkpoint may contain training state, extract model weights
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

# Prepare tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load and prepare data
df = pd.concat([
    pd.read_csv('Fake_True/Fake.csv').assign(label=1).head(100),
    pd.read_csv('Fake_True/True.csv').assign(label=0).head(100)
], ignore_index=True)

texts = df['title'].astype(str).tolist() if 'title' in df.columns else df[df.columns[0]].astype(str).tolist()
labels = df['label'].values

# Split into validation set (20% as specified)
text_train, text_val, label_train, label_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Tokenize validation data
val_encodings = tokenizer(
    text_val, 
    truncation=True, 
    max_length=128, 
    padding=True, 
    return_tensors='pt'
)

val_input_ids = val_encodings['input_ids'].to(device)
val_attention_mask = val_encodings['attention_mask'].to(device)
val_labels = torch.tensor(label_val, dtype=torch.long).to(device)

# Create DataLoader
val_dataset = TensorDataset(val_input_ids, val_attention_mask, val_labels)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Run validation with attention extraction
all_predictions = []
all_labels = []
all_attention_weights = []

with torch.no_grad():
    for batch_input_ids, batch_att_mask, batch_labels in val_loader:
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_att_mask,
            output_attentions=True
        )
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        attentions = outputs.attentions  # Tuple of (batch_size, num_heads, seq_len, seq_len) for each layer
        
        all_predictions.extend(predictions)
        all_labels.extend(batch_labels.cpu().numpy())
        
        # Store average attention from last layer
        last_layer_attn = attentions[-1]  # (batch_size, num_heads, seq_len, seq_len)
        all_attention_weights.append(last_layer_attn.cpu().numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Combine attention weights
all_attention_weights = np.concatenate(all_attention_weights, axis=0)  # (total_samples, num_heads, seq_len, seq_len)

# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
cm = confusion_matrix(all_labels, all_predictions)
print(f"\nValidation Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:\n{classification_report(all_labels, all_predictions, target_names=['Real', 'Fake'])}")
print(f"\nConfusion Matrix:\n{cm}")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confusion Matrix (Normalized)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=True)
axes[0, 0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')
axes[0, 0].set_xticklabels(['Real', 'Fake'])
axes[0, 0].set_yticklabels(['Real', 'Fake'])

# 2. Normalized Confusion Matrix
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=axes[0, 1], cbar=True)
axes[0, 1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('True Label')
axes[0, 1].set_xlabel('Predicted Label')
axes[0, 1].set_xticklabels(['Real', 'Fake'])
axes[0, 1].set_yticklabels(['Real', 'Fake'])

# 3. Average attention pattern across all samples and heads
avg_attention = all_attention_weights.mean(axis=(0, 1))  # Average over samples and heads
im = axes[1, 0].imshow(avg_attention, cmap='viridis', aspect='auto')
axes[1, 0].set_title('Average Attention Pattern (All Layers, All Heads)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Attended Position')
axes[1, 0].set_ylabel('Query Position')
plt.colorbar(im, ax=axes[1, 0])

# 4. Attention from [CLS] token (first token - typically used for classification)
cls_attention = all_attention_weights.mean(axis=1)[:, 0, :]  # Average over heads, take first token attention
im = axes[1, 1].imshow(cls_attention.T, cmap='hot', aspect='auto')
axes[1, 1].set_title('[CLS] Token Attention (What it attends to)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Position in Sequence')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print(f"\nValidation set size: {len(all_labels)}")
print(f"Number of attention heads: {all_attention_weights.shape[1]}")
print(f"Sequence length: {all_attention_weights.shape[-1]}")
